In [ ]:
# | default_exp coverage

# Coverage analysis

> Per-tile and per-obsid fractional coverage of dark deposits (union of fan + blotch polygons / tile pixel area). Verbatim port of Tom Ihro's `Calculate_Coverage_v2.ipynb` (cells 0–28).

Note: this is *analysis* of the catalog, not part of catalog production. Lives at top level (`p4tools.coverage`) since v0.19; the legacy `p4tools.production.coverage` import path is preserved as a deprecation shim until v0.20.

In [ ]:
# | export
"""coverage — per-tile and per-obsid fractional coverage of dark deposits.

Port of Tom Ihro's ``Calculate_Coverage_v2.ipynb`` for v3.1, packaged as a
reproducible p4tools-native module. Inputs are the v3.1 fan and blotch
catalogs (fetched via ``p4tools.io.get_fan_catalog`` / ``get_blotch_catalog``);
outputs are the per-tile and per-obsid coverage tables previously distributed
only as Tom's local CSVs.

The fractional coverage produced here is **dimensionless** (polygon area in
pixel² divided by tile area in pixel²); ``map_scale`` cancels out so values
are directly comparable across observations of different ground resolutions.
For *absolute* metrics (markings per m², per-marking area in m²), see
:mod:`p4tools.activity`.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.stats as _sps

from p4tools import io as _io
from p4tools import markings as _mk

TILE_WIDTH_PX = 840
"""P4 tile width in HiRISE pixels."""

TILE_HEIGHT_PX = 648
"""P4 tile height in HiRISE pixels."""

TILE_AREA_PX = TILE_WIDTH_PX * TILE_HEIGHT_PX  # 544 320

DEFAULT_CACHE_DIR = Path("~/.cache/p4tools").expanduser()


def _row_to_fan(row):
    return _mk.Fan(row.copy(), scope="hirise")


def _row_to_blotch(row):
    return _mk.Blotch(row.copy(), scope="hirise")


## Per-tile coverage

In [ ]:
# | export
def compute_per_tile_coverage(
    version: str = "v3.1",
    *,
    fan: pd.DataFrame | None = None,
    blotch: pd.DataFrame | None = None,
    cache: bool = True,
    cache_dir: Path | None = None,
) -> pd.DataFrame:
    """Per-tile fractional dark-deposit coverage.

    Verbatim port of Tom Ihro's ``Calculate_Coverage_v2.ipynb`` cells 0–20.
    Result has columns ``[obsid, tile_id, Coverage]``; for v3.1 it has
    ~64 494 rows. Cached as parquet so the (slow) Shapely union runs once
    per machine.
    """
    if cache_dir is None:
        cache_dir = DEFAULT_CACHE_DIR
    cache_dir = Path(cache_dir)
    cache_path = cache_dir / f"FnotchCoverage_Full_{version}.parquet"
    if cache and cache_path.exists():
        return pd.read_parquet(cache_path)

    fan_df = fan if fan is not None else _io.get_fan_catalog(version)
    blotch_df = blotch if blotch is not None else _io.get_blotch_catalog(version)

    fans = fan_df.apply(_row_to_fan, axis=1)
    geom_fan = fans.apply(_mk.Fan.to_shapely)
    fan_gpd = gpd.GeoDataFrame(fan_df, geometry=geom_fan)

    blotches = blotch_df.apply(_row_to_blotch, axis=1)
    geom_blotch = blotches.apply(_mk.Blotch.to_shapely)
    blotch_gpd = gpd.GeoDataFrame(blotch_df, geometry=geom_blotch)

    gpd_comb = pd.concat([fan_gpd, blotch_gpd])
    dissolved = gpd_comb.dissolve(["obsid", "tile_id"])
    coverage = (dissolved.area / TILE_AREA_PX).rename("Coverage").reset_index()
    out = coverage[["obsid", "tile_id", "Coverage"]].copy()

    if cache:
        cache_dir.mkdir(parents=True, exist_ok=True)
        out.to_parquet(cache_path, index=False)
    return out


## Per-obsid summary

In [ ]:
# | export
def compute_per_obsid_coverage(
    per_tile: pd.DataFrame,
    *,
    with_homogeneity: bool = True,
) -> pd.DataFrame:
    """Aggregate per-tile coverage to per-obsid summary statistics.

    Verbatim port of cells 26–28 of ``Calculate_Coverage_v2.ipynb``:
    mean, median, std, skew, kurtosis (and optional Homogeneity = mean / median).
    """
    agg = per_tile.groupby("obsid").agg(
        {"Coverage": ["mean", "median", "std", "skew", _sps.kurtosis]}
    )
    agg.columns = ["_".join(c) for c in agg.columns.values]
    if with_homogeneity:
        agg["Homogeneity"] = agg["Coverage_mean"] / agg["Coverage_median"]
    return agg.reset_index()


In [ ]:
# | export
# | eval: false
if __name__ == "__main__":
    df = compute_per_tile_coverage()
    print(df.head())
